## MMDS Final Project Part 1.

In [1]:
import random
random.seed(42)

DATA_DIR = "./data"
file = f"{DATA_DIR}/ratings.dat" # MovieLens 1M analog to the `u.data` from MovieLens 100k, which was mentioned in the task.

## 1. Stream random sampling

The following code cell creates a random sample of 10% of rated movies from the given dataset.

In [98]:
random_sample = []

with open(file, "r") as f:
    for line in f:
        user_id, movie_id, rating, timestamp = line.strip('\n').split('::')

        if hash(str(movie_id)) % 10 == 0:
            random_sample.append((user_id, movie_id, rating, timestamp))

print(f"Created a sample of {len(random_sample)} ratings for 10% of rated movies.")

Created a sample of 106389 ratings for 10% of rated movies.


## 2. Counting distinct elements

The following code cell counts distinct users that left movie ratings.

In [100]:
def trailing_zeros(num):
  if num == 0:
    return 32 # Assuming ints are 32bit.
  p = 0
  while (num >> p) & 1 == 0:
    p += 1
  return p

# Ground truth via a naive approach.
with open(file, "r") as f:
    naive = set(line.strip('\n').split('::')[0] for line in f)
    naive = len(naive)


flaj_martin = 0

bucket_bits = 10
num_buckets = 1 << bucket_bits
buckets = [0] * num_buckets

# Probabilistic counting via Flajolet-Martin and LogLog.
with open(file, "r") as f:
    for line in f:
        user_id, movie_id, rating, timestamp = line.strip('\n').split('::')
        h = hash(str(user_id))

        # Original Flajolet-Martin. Heavily affected by outliers -> unstable.
        flaj_martin = max(flaj_martin, trailing_zeros(h))

        # LogLog. Averages across different buckets.
        bucket = h & (num_buckets - 1)
        buckets[bucket] = max(buckets[bucket], trailing_zeros(h >> bucket_bits))

avg = sum(buckets) / num_buckets
loglog = num_buckets * (2 ** avg) * 0.77351  # 0.77351 is a bias correction factor

print("Distinct user count estimates")
print("Truth:\t", naive)
print("FM:\t", 2**flaj_martin)
print("LogLog:\t", int(loglog))

Distinct user count estimates
Truth:	 6040
FM:	 32768
LogLog:	 6146
